In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

df = pd.read_csv("train.csv")   # root of the project folder
print(df.shape)
df.head()

(7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
print((df['target']).value_counts())
print(df['text'].isna().sum())

target
0    4342
1    3271
Name: count, dtype: int64
0


In [6]:
train_df , val_df= train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=42, 
)

print ("Train:" , train_df.shape, "Val:" , val_df.shape)

Train: (6090, 5) Val: (1523, 5)


In [7]:
def evaluate(y_true, y_pred, model_name="model"):
    metrics = {
        "model": model_name,
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    print(f"[{model_name}]  F1={metrics['f1']:.4f}  P={metrics['precision']:.4f}  "
          f"R={metrics['recall']:.4f}  Acc={metrics['accuracy']:.4f}")
    return metrics

results = []

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,       # "Fire" and "fire" become the same feature
    stop_words="english", # drop the/a/is/... — IDF already suppresses them, this just makes it explicit + smaller
    ngram_range=(1, 1),   # unigrams only for the pure baseline (single words)
    min_df=2,             # ignore words appearing in only 1 tweet — almost always noise/typos
)

# CRITICAL: fit ONLY on train. The vocabulary and the IDF statistics are LEARNED,
# so learning them from validation data would be leakage — the model would have
# "seen" val while training. fit_transform on train, transform (no fit) on val.
X_train = vectorizer.fit_transform(train_df["text"])
X_val   = vectorizer.transform(val_df["text"])

y_train = train_df["target"].values
y_val   = val_df["target"].values

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("X_train shape:", X_train.shape)   # (n_train, vocab_size), sparse

Vocabulary size: 5467
X_train shape: (6090, 5467)


In [9]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 46439 stored elements and shape (6090, 5467)>
  Coords	Values
  (0, 4188)	0.31944228803396985
  (0, 1023)	0.21367907511545353
  (0, 2098)	0.2305387214414275
  (0, 1209)	0.2510070323501371
  (0, 2399)	0.31944228803396985
  (0, 4594)	0.30050308732123077
  (0, 4430)	0.31944228803396985
  (0, 3199)	0.2650768631876816
  (0, 4438)	0.24729637137024835
  (0, 365)	0.31944228803396985
  (0, 2384)	0.06484615639005191
  (0, 2418)	0.31944228803396985
  (0, 698)	0.31944228803396985
  (1, 2384)	0.17434263075944265
  (1, 2119)	0.2882120556108197
  (1, 2682)	0.41508142152750704
  (1, 2286)	0.3948730266274511
  (1, 2141)	0.3746646317273952
  (1, 4138)	0.3805349422657723
  (1, 3565)	0.23280989917836808
  (1, 1584)	0.3492050651119204
  (1, 403)	0.31144198374230286
  (2, 2612)	0.49406856890713746
  (2, 3771)	0.44596074123593793
  (2, 2053)	0.37453595642329296
  :	:
  (6086, 930)	0.3784670424230236
  (6086, 4046)	0.36583023317715596
  (6087, 403)

In [10]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)  # 1000 iters so it fully converges on sparse text
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)           # 0/1 predictions
metrics = evaluate(y_val, val_preds, model_name="TF-IDF + LogReg")  # our Phase 0 harness
results.append(metrics)  

[TF-IDF + LogReg]  F1=0.7645  P=0.8351  R=0.7049  Acc=0.8135


In [11]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]  # one weight per vocabulary word

top_disaster = np.argsort(coefs)[-20:][::-1]   # 20 most positive → push toward "disaster"
top_normal   = np.argsort(coefs)[:20]          # 20 most negative → push toward "not"

print("Words most predictive of DISASTER:")
print(feature_names[top_disaster])
print("\nWords most predictive of NOT-disaster:")
print(feature_names[top_normal])

Words most predictive of DISASTER:
['hiroshima' 'http' 'california' 'fires' 'buildings' 'killed' 'storm'
 'suicide' 'wildfire' 'typhoon' 'train' 'bombing' 'police' 'near'
 'earthquake' 'forest' 'japan' 'mass' 'floods' 'massacre']

Words most predictive of NOT-disaster:
['love' 'new' 'harm' 'nowplaying' 'wrecked' 'ruin' 'body' 'explode'
 'stretcher' 'bloody' 'let' 'traumatised' 'bags' 'ebay' 'demolish'
 'blazing' 'know' 'blew' 'self' 'drown']


In [12]:
import os, urllib.request, zipfile

# GloVe 6B, 100-dimensional. ~130MB zipped. Downloads once, then cached locally.
if not os.path.exists("glove.6B.100d.txt"):
    url = "https://nlp.stanford.edu/data/glove.6B.zip"
    print("Downloading GloVe (~800MB zip, contains several dims)...")
    urllib.request.urlretrieve(url, "glove.6B.zip")
    with zipfile.ZipFile("glove.6B.zip") as z:
        z.extract("glove.6B.100d.txt")  # extract only the 100d file
    print("Done.")

# Load into a dict: word -> numpy vector of 100 floats
import numpy as np
glove = {}
with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.asarray(parts[1:], dtype="float32")
print("GloVe words loaded:", len(glove))  # ~400,000

GloVe words loaded: 400000


In [13]:
import re
from collections import Counter

def tokenize(text):
    # lowercase, keep words only. Simple on purpose — tweets are messy;
    # a heavier clean is a possible experiment later.
    return re.findall(r"[a-z]+", text.lower())

# Count words in TRAIN ONLY (same leakage logic as Phase 1 — vocab is learned from train).
counter = Counter()
for t in train_df["text"]:
    counter.update(tokenize(t))

# Keep words seen at least twice. Reserve 0=<pad>, 1=<unk>.
itos = ["<pad>", "<unk>"] + [w for w, c in counter.items() if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
vocab_size = len(itos)
print("Vocab size:", vocab_size)

# Build the embedding matrix: row i = GloVe vector for word itos[i].
EMB_DIM = 100
embedding_matrix = np.zeros((vocab_size, EMB_DIM), dtype="float32")
hits, misses = 0, 0
for i, word in enumerate(itos):
    if word in glove:
        embedding_matrix[i] = glove[word]   # use GloVe's learned vector
        hits += 1
    else:
        # word not in GloVe (rare/slang/typo) -> small random vector, will be learned
        embedding_matrix[i] = np.random.normal(scale=0.6, size=(EMB_DIM,))
        misses += 1
print(f"GloVe hits: {hits}, misses: {misses}")  # expect most words to hit

Vocab size: 6119
GloVe hits: 5722, misses: 397


In [14]:
import torch

MAX_LEN = 30  # covers virtually all tweets; longer ones truncated

def encode(text):
    ids = [stoi.get(w, 1) for w in tokenize(text)]  # word->id, unknown->1
    ids = ids[:MAX_LEN]                              # truncate
    ids = ids + [0] * (MAX_LEN - len(ids))           # pad with 0 to MAX_LEN
    return ids

X_train_ids = torch.tensor([encode(t) for t in train_df["text"]], dtype=torch.long)
X_val_ids   = torch.tensor([encode(t) for t in val_df["text"]],   dtype=torch.long)
y_train_t   = torch.tensor(train_df["target"].values, dtype=torch.float32)
y_val_t     = torch.tensor(val_df["target"].values,   dtype=torch.float32)

print(X_train_ids.shape)  # (n_train, 30)

torch.Size([6090, 30])


In [15]:
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape

        # Embedding layer = the lookup table. padding_idx=0 keeps <pad> at zero.
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        # Load GloVe into it. freeze=False -> let it fine-tune slightly during training.
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        # The BiLSTM. bidirectional=True runs two LSTMs (both directions).
        # batch_first -> input shape is (batch, seq, features).
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)

        self.dropout = nn.Dropout(0.3)  # regularization — small dataset, easy to overfit
        # hidden_dim*2 because bidirectional concatenates both directions' final states.
        self.fc = nn.Linear(hidden_dim * 2, 1)  # -> single logit for binary classification

    def forward(self, x):
        emb = self.embedding(x)                 # (batch, seq, emb_dim)
        outputs, (h_n, c_n) = self.lstm(emb)    # h_n: final hidden states
        # h_n shape: (2, batch, hidden) — [0]=forward last, [1]=backward last.
        # Concatenate both directions' final hidden states:
        h = torch.cat([h_n[0], h_n[1]], dim=1)  # (batch, hidden*2)
        h = self.dropout(h)
        return self.fc(h).squeeze(1)            # (batch,) raw logits

In [16]:
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)  # "cpu" is fine here — this model is small

train_ds = TensorDataset(X_train_ids, y_train_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

model = BiLSTMClassifier(embedding_matrix).to(device)
criterion = nn.BCEWithLogitsLoss()  # binary cross-entropy, takes raw logits (numerically stable)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()          # clear old gradients
        logits = model(xb)             # forward pass
        loss = criterion(logits, yb)   # how wrong
        loss.backward()                # backprop (through time!)
        optimizer.step()               # update weights
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={total_loss/len(train_dl):.4f}")

Device: cpu
Epoch 1/10  loss=0.5155
Epoch 2/10  loss=0.4040
Epoch 3/10  loss=0.3593
Epoch 4/10  loss=0.3126
Epoch 5/10  loss=0.2669
Epoch 6/10  loss=0.2168
Epoch 7/10  loss=0.1817
Epoch 8/10  loss=0.1460
Epoch 9/10  loss=0.1198
Epoch 10/10  loss=0.0999


In [17]:
model.eval()
with torch.no_grad():   # no gradients needed for inference — faster, less memory
    val_logits = model(X_val_ids.to(device))
    val_probs = torch.sigmoid(val_logits)          # logits -> probabilities
    val_preds = (val_probs > 0.5).long().cpu().numpy()  # threshold at 0.5 -> 0/1

metrics = evaluate(y_val_t.numpy().astype(int), val_preds, model_name="GloVe + BiLSTM")
results.append(metrics)  # same harness, same results list -> honest comparison

[GloVe + BiLSTM]  F1=0.7600  P=0.7776  R=0.7431  Acc=0.7984


In [18]:
# run once in the terminal (or prefix with ! in a notebook cell):
# pip install transformers

import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [20]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Inspect what it actually does to a real tweet
sample = "Forest fire near La Ronge Sask. Canada"
enc = tokenizer(sample)

print("Tokens:   ", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("Input IDs:", enc["input_ids"])
print("Mask:     ", enc["attention_mask"])
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokens:    ['[CLS]', 'forest', 'fire', 'near', 'la', 'ron', '##ge', 'sas', '##k', '.', 'canada', '[SEP]']
Input IDs: [101, 3224, 2543, 2379, 2474, 6902, 3351, 21871, 2243, 1012, 2710, 102]
Mask:      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Vocab size: 30522


In [21]:
MAX_LEN = 64

def encode_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,          # cut anything over MAX_LEN
        padding="max_length",     # pad everything up to MAX_LEN
        max_length=MAX_LEN,
        return_tensors="pt",      # give me PyTorch tensors directly
    )

train_enc = encode_texts(train_df["text"])
val_enc   = encode_texts(val_df["text"])

y_train_t = torch.tensor(train_df["target"].values, dtype=torch.long)
y_val_t   = torch.tensor(val_df["target"].values,   dtype=torch.long)

train_ds = TensorDataset(train_enc["input_ids"], train_enc["attention_mask"], y_train_t)
val_ds   = TensorDataset(val_enc["input_ids"],   val_enc["attention_mask"],   y_val_t)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=32)   # bigger batch fine, no gradients at eval

print(train_enc["input_ids"].shape)   # (n_train, 64)

torch.Size([6090, 64])


In [22]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,            # binary: not-disaster / disaster
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.1f}M")   # ~67M

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 67.0M


In [23]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for input_ids, attn_mask, labels in train_dl:
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels    = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attn_mask,
                        labels=labels)      # passing labels -> HF computes the loss for us
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # stability
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}  loss={total_loss/len(train_dl):.4f}")

Epoch 1/3  loss=0.4415
Epoch 2/3  loss=0.3349
Epoch 3/3  loss=0.2553


In [24]:
model.eval()
all_preds = []

with torch.no_grad():
    for input_ids, attn_mask, labels in val_dl:
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        preds = torch.argmax(logits, dim=1)     # 2 logits -> pick the higher
        all_preds.append(preds.cpu())

val_preds = torch.cat(all_preds).numpy()

metrics = evaluate(y_val_t.numpy(), val_preds, model_name="DistilBERT fine-tuned")
results.append(metrics)

[DistilBERT fine-tuned]  F1=0.7994  P=0.7916  R=0.8073  Acc=0.8260
